# MultiModalSCVI — Blinatumomab annotated data

Train 3 models: joint POE (abundance + top-500 arsinh spatial), abundance-only, spatial-only.
Benchmark: loss curves, UMAPs, PPC (joint), simple scib (bio=`cell_type_annot`, batch=`cell_system`).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import torch
import scvi
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA

from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.enums import AggMethod, D
from PixelGen.pxl_utils import train_model
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap
from PixelGen.utils import plot_composite_ppc, get_dense, calculate_metrics

scvi.settings.seed = 0
torch.set_float32_matmul_precision('high')
sc.set_figure_params(figsize=(5, 3), frameon=False)

NEW_DATA = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data')
CACHE    = NEW_DATA / 'cache'
MODELS   = CACHE / 'models'
MODELS.mkdir(parents=True, exist_ok=True)
print('scvi', scvi.__version__, '| cuda', torch.cuda.is_available())
print('cache:', CACHE)

In [ ]:
adata = sc.read_h5ad(CACHE / 'adata_cytovi_annotated_compat.h5ad')
print(adata)
print('arcsinh layer:', adata.layers['arcsinh'].shape)
print('spatial_asinh5_top500var:', adata.obsm['spatial_asinh5_top500var'].shape)
print('\ncell_type_annot:'); print(adata.obs['cell_type_annot'].value_counts())
print('\ncell_system:');     print(adata.obs['cell_system'].value_counts())

In [ ]:
ABUNDANCE_LAYER = 'arcsinh'
SPATIAL_KEY     = 'spatial_asinh5_top500var'
BATCH_KEY       = 'cell_system'
BIO_KEY         = 'cell_type_annot'

base_train_kwargs = dict(
    train_size=0.8,
    check_val_every_n_epoch=1,
    early_stopping=True,
    early_stopping_patience=200,
    batch_size=2000,
    max_epochs=10000,
    enable_checkpointing=True,
    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400),
)

In [ ]:
SPATIAL_KEY_TOP50 = 'spatial_asinh5_top50var'
sp500 = adata.obsm[SPATIAL_KEY]
top50 = sp500.var().sort_values(ascending=False).head(50).index
adata.obsm[SPATIAL_KEY_TOP50] = sp500[top50].copy()
print(adata.obsm[SPATIAL_KEY_TOP50].shape, '| top features:')
print(list(top50[:10]))

In [ ]:
from PixelGen.utils import build_spatial_celltype_obsm
# Cell-type-aware selection: per cell type sum significant |z|>2.04 across cells, top-100 pairs/type, union.
SPATIAL_KEY_CT = build_spatial_celltype_obsm(
    adata, celltype_key=BIO_KEY, score_key='spatial_raw', value_key='spatial_asinh5',
    z_thresh=2.04, top_x=100, use_abs=True,
)  # -> 'spatial_asinh5_ct_top100'
print(adata.obsm[SPATIAL_KEY_CT].shape)
for ct, pairs in adata.uns[f'{SPATIAL_KEY_CT}_info']['per_celltype_pairs'].items():
    print(f'{ct:10s}', pairs[:8])

## Model A — joint POE (abundance + spatial)

In [ ]:
setup_kwargs_joint = dict(
    layer=ABUNDANCE_LAYER,
    extra_modality_keys=[SPATIAL_KEY],
    n_modalities=2,
    batch_key=BATCH_KEY,
    spatial_mask_key=None,
)
model_kwargs_joint = dict(
    n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal, D.Normal],
    experts_method='POE',
    loss_weights='auto',
    joint_kl=False, unimodal_kl=True,
    batch_mask=[False, True],  # batch-corr spatial only
)
model_joint = train_model(adata, MultiModalSCVI, setup_kwargs_joint, model_kwargs_joint, base_train_kwargs)
model_joint.save(str(MODELS / 'poe_joint'), overwrite=True)
print('joint weights:', model_joint.get_weights())

## Model B — abundance-only

In [ ]:
setup_kwargs_abn = dict(
    layer=ABUNDANCE_LAYER, extra_modality_keys=[], n_modalities=1,
    batch_key=BATCH_KEY, spatial_mask_key=None,
)
model_kwargs_abn = dict(
    n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal], unimodal_kl=True, joint_kl=False,
)
model_abn = train_model(adata, MultiModalSCVI, setup_kwargs_abn, model_kwargs_abn, base_train_kwargs)
model_abn.save(str(MODELS / 'abundance_only'), overwrite=True)

## Model C — spatial-only
Spatial features (`spatial_asinh5_top500var`, shape n×500) live in `obsm`, but the setup uses the AnnData's main layer for the single modality. Build a sidecar AnnData with X = spatial values.

In [ ]:
sp_df = adata.obsm[SPATIAL_KEY]
adata_spt = ad.AnnData(
    X=sp_df.values.astype(np.float32),
    obs=adata.obs.copy(),
    var=pd.DataFrame(index=sp_df.columns.astype(str)),
)
adata_spt.layers['spatial'] = adata_spt.X.copy()

setup_kwargs_spt = dict(
    layer='spatial', extra_modality_keys=[], n_modalities=1,
    batch_key=BATCH_KEY, spatial_mask_key=None,
)
model_kwargs_spt = dict(
    n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal], unimodal_kl=True, joint_kl=False,
)
model_spt = train_model(adata_spt, MultiModalSCVI, setup_kwargs_spt, model_kwargs_spt, base_train_kwargs)
model_spt.save(str(MODELS / 'spatial_only'), overwrite=True)

## Model D — joint POE (abundance + spatial top 50)

In [ ]:
setup_kwargs_joint50 = dict(
    layer=ABUNDANCE_LAYER,
    extra_modality_keys=[SPATIAL_KEY_TOP50],
    n_modalities=2,
    batch_key=BATCH_KEY,
    spatial_mask_key=None,
)
model_kwargs_joint50 = dict(
    n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal, D.Normal],
    experts_method='POE',
    loss_weights='auto',
    joint_kl=False, unimodal_kl=True,
    batch_mask=[False, True],
)
model_joint50 = train_model(adata, MultiModalSCVI, setup_kwargs_joint50, model_kwargs_joint50, base_train_kwargs)
model_joint50.save(str(MODELS / 'poe_joint_top50'), overwrite=True)
print('joint50 weights:', model_joint50.get_weights())

## Model E — joint POE (abundance + spatial cell-type top-100)
Spatial modality = pairs selected by per-cell-type summed significant |z| (union of top-100/type).

In [ ]:
setup_kwargs_ct = dict(
    layer=ABUNDANCE_LAYER,
    extra_modality_keys=[SPATIAL_KEY_CT],
    n_modalities=2,
    batch_key=BATCH_KEY,
    spatial_mask_key=None,
)
model_kwargs_ct = dict(
    n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal, D.Normal],
    experts_method='POE',
    loss_weights='auto',
    joint_kl=False, unimodal_kl=True,
    batch_mask=[False, True],
)
model_ct = train_model(adata, MultiModalSCVI, setup_kwargs_ct, model_kwargs_ct, base_train_kwargs)
model_ct.save(str(MODELS / 'poe_joint_ct_top100'), overwrite=True)
print('ct weights:', model_ct.get_weights())

## Loss curves

In [ ]:
for name, m in [('poe_joint', model_joint),
                ('poe_joint_top50', model_joint50),
                ('poe_joint_ct_top100', model_ct),
                ('abundance_only', model_abn),
                ('spatial_only', model_spt)]:
    print(name)
    plot_losses(m)
    plt.show()

## Latents

In [ ]:
adata.obsm['z_joint']        = model_joint.get_latent_representation(adata, modality='joint')
adata.obsm['z_joint_abn']    = model_joint.get_latent_representation(adata, modality=ABUNDANCE_LAYER)
adata.obsm['z_joint_spt']    = model_joint.get_latent_representation(adata, modality=SPATIAL_KEY)
adata.obsm['z_joint_top50']  = model_joint50.get_latent_representation(adata, modality='joint')
adata.obsm['z_joint_ct']     = model_ct.get_latent_representation(adata, modality='joint')
adata.obsm['z_abn_only']     = model_abn.get_latent_representation(adata, modality='joint')
adata.obsm['z_spt_only']     = model_spt.get_latent_representation(adata_spt, modality='joint')

X_concat = np.concatenate([
    get_dense(adata.layers[ABUNDANCE_LAYER]),
    get_dense(adata.obsm[SPATIAL_KEY]),
], axis=1)
adata.obsm['X_pca_concat'] = PCA(n_components=30, random_state=0).fit_transform(X_concat)

for k in ['z_joint', 'z_joint_abn', 'z_joint_spt', 'z_joint_top50', 'z_joint_ct', 'z_abn_only', 'z_spt_only', 'X_pca_concat']:
    print(f'{k:20s} {adata.obsm[k].shape}')

## UMAPs

In [ ]:
for key, title in [
    ('z_joint',       'POE joint (spatial top500)'),
    ('z_joint_top50', 'POE joint (spatial top50)'),
    ('z_joint_ct',    'POE joint (spatial ct-top100)'),
    ('z_joint_abn',   'POE abundance head'),
    ('z_joint_spt',   'POE spatial head'),
    ('z_abn_only',    'Abundance-only'),
    ('z_spt_only',    'Spatial-only'),
    ('X_pca_concat',  'PCA concat baseline'),
]:
    pca_neighbors_umap(
        adata, latent_name=key,
        umap_pl_kwargs=dict(color=[BIO_KEY, BATCH_KEY], frameon=False, ncols=2),
        umap_title=title,
    )
    plt.show()

## Spatial autocorrelation benchmark

For each model's latent we build a kNN graph and compute Moran's I on every abundance and spatial feature. A latent that better organizes cells along biologically/spatially meaningful axes yields *higher* Moran's I — features become smoother across neighbouring cells. Reps: abundance = `arcsinh` (layer), spatial = `spatial_asinh5_top500var` (same ground-truth for every model).

In [ ]:
from PixelGen.metrics import distr_autocorrelation_in_latent

# Latents already stored on adata.obsm (see Latents cell). Same kNN graph per latent,
# Moran's I computed on abundance + spatial reps.
ac_latent_keys = [
    'z_joint', 'z_joint_top50', 'z_joint_ct', 'z_joint_abn', 'z_joint_spt',
    'z_abn_only', 'z_spt_only', 'X_pca_concat',
]
ac_latent_names = [
    'poe_joint_top500', 'poe_joint_top50', 'poe_joint_ct', 'poe_abn_head', 'poe_spt_head',
    'abundance_only', 'spatial_only', 'pca_concat',
]

autocorr_abundance = distr_autocorrelation_in_latent(
    adata, latent_keys=ac_latent_keys, names=ac_latent_names,
    rep_key=ABUNDANCE_LAYER, pca_kwargs={'n_comps': 15},
)
autocorr_spatial = distr_autocorrelation_in_latent(
    adata, latent_keys=ac_latent_keys, names=ac_latent_names,
    rep_key=SPATIAL_KEY, pca_kwargs={'n_comps': 15},
)
print('abundance:', autocorr_abundance.shape, '| spatial:', autocorr_spatial.shape)
autocorr_spatial.head()

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=autocorr_spatial, x='morans', hue='latent',
             kde=True, stat='density', common_norm=False, alpha=0.4, ax=axes[0])
axes[0].set_title(f"Spatial Moran's I  (rep={SPATIAL_KEY}, {adata.obsm[SPATIAL_KEY].shape[1]} pairs)")
axes[0].set_xlabel("Moran's I")

sns.histplot(data=autocorr_abundance, x='morans', hue='latent',
             kde=True, stat='density', common_norm=False, alpha=0.4, ax=axes[1])
axes[1].set_title(f"Abundance Moran's I  (rep={ABUNDANCE_LAYER}, {adata.layers[ABUNDANCE_LAYER].shape[1]} markers)")
axes[1].set_xlabel("Moran's I")
plt.tight_layout(); plt.show()

In [ ]:
# Summary stats (mean / median of Moran's I across features) per model and feature type.
summary_rows = []
for label, df in [('Spatial', autocorr_spatial), ('Abundance', autocorr_abundance)]:
    for name in ac_latent_names:
        s = df[df['latent'] == name]['morans']
        summary_rows.append({'Feature type': label, 'Model': name,
                             'Mean': s.mean(), 'Median': s.median(), 'Std': s.std()})
summary_df = pd.DataFrame(summary_rows)
summary_df.pivot(index='Model', columns='Feature type', values=['Mean', 'Median'])

In [ ]:
# Paired scatter: Moran's I per feature, abundance_only baseline vs each other latent.
BASELINE = 'abundance_only'
comparisons = [name for name in ac_latent_names if name != BASELINE]
fig, axes = plt.subplots(len(comparisons), 2, figsize=(11, 4.5 * len(comparisons)))
if len(comparisons) == 1:
    axes = axes.reshape(1, 2)
for row, name in enumerate(comparisons):
    for col, (label, df) in enumerate([('Spatial', autocorr_spatial), ('Abundance', autocorr_abundance)]):
        ax = axes[row, col]
        x = df[df['latent'] == BASELINE]['morans'].values
        y = df[df['latent'] == name]['morans'].values
        ax.scatter(x, y, alpha=0.4, s=12)
        lo = min(x.min(), y.min()); hi = max(x.max(), y.max())
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.5)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
        ax.set_xlabel(f"Moran's I — {BASELINE}")
        ax.set_ylabel(f"Moran's I — {name}")
        n_above = int((y > x).sum()); n = len(x)
        ax.set_title(f'{label}   ({n_above}/{n} above diag)')
plt.tight_layout(); plt.show()

## PPC — POE top500 vs POE top50
Each model reconstructs its own input. Abundance is the same `arcsinh` matrix for both, directly comparable. Spatial uses each model's own feature set (500 vs 50) — measures self-reconstruction quality, not a head-to-head on identical features.

In [ ]:
def _ppc_one(model, abn_layer, sp_key):
    out = model.get_normalized_expression(
        adata=adata,
        return_mean_expression=True,
        return_l2_error=True,
        return_px_distrs=False,
        return_numpy=True,
    )
    return (
        get_dense(adata.layers[abn_layer]),  get_dense(out['exprs'][abn_layer]),
        get_dense(adata.obsm[sp_key]),       get_dense(out['exprs'][sp_key]),
    )

ab_raw_500, ab_gen_500, sp_raw_500, sp_gen_500 = _ppc_one(model_joint,   ABUNDANCE_LAYER, SPATIAL_KEY)
ab_raw_50,  ab_gen_50,  sp_raw_50,  sp_gen_50  = _ppc_one(model_joint50, ABUNDANCE_LAYER, SPATIAL_KEY_TOP50)
ab_raw_ct,  ab_gen_ct,  sp_raw_ct,  sp_gen_ct  = _ppc_one(model_ct,      ABUNDANCE_LAYER, SPATIAL_KEY_CT)

model_names = ['poe_joint_top500', 'poe_joint_top50', 'poe_joint_ct_top100']
all_metrics  = []
all_metrics += calculate_metrics(ab_raw_500, ab_gen_500, 'poe_joint_top500',    'Abundance')
all_metrics += calculate_metrics(ab_raw_50,  ab_gen_50,  'poe_joint_top50',     'Abundance')
all_metrics += calculate_metrics(ab_raw_ct,  ab_gen_ct,  'poe_joint_ct_top100', 'Abundance')
all_metrics += calculate_metrics(sp_raw_500, sp_gen_500, 'poe_joint_top500',    'Spatial')
all_metrics += calculate_metrics(sp_raw_50,  sp_gen_50,  'poe_joint_top50',     'Spatial')
all_metrics += calculate_metrics(sp_raw_ct,  sp_gen_ct,  'poe_joint_ct_top100', 'Spatial')
metrics_df = pd.DataFrame(all_metrics)

plot_composite_ppc('Abundance', metrics_df, 'cornflowerblue', model_names); plt.show()
plot_composite_ppc('Spatial',   metrics_df, 'lightgreen',     model_names); plt.show()

## scib metrics (simple)
bio = `cell_type_annot`, batch = `cell_system`.

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

embedding_keys = ['z_joint', 'z_joint_top50', 'z_joint_ct', 'z_joint_abn', 'z_joint_spt',
                  'z_abn_only', 'z_spt_only', 'X_pca_concat']
bm = Benchmarker(
    adata,
    batch_key=BATCH_KEY,
    label_key=BIO_KEY,
    embedding_obsm_keys=embedding_keys,
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
)
bm.benchmark()
bm.plot_results_table(min_max_scale=False)

## Model F — joint POE (abundance + pairwise spatial + 3-way triplet)

Adds a **3-way colocalization** modality built from the pairwise `spatial_raw` z-scores via the 3×3 correlation-matrix determinant (`utils.build_triplet_obsm`, `metric='determinant'`). No model-code change: `setup_anndata` reads the new obsm's width automatically, so it registers as a 3rd POE modality alongside abundance and pairwise spatial.

In [ ]:
import json
from PixelGen.utils import build_triplet_obsm, get_marker_triplets

# 3-way colocalization derived from the pairwise z-scores already in obsm['spatial_raw'].
# Restrict to a curated panel intersected with markers actually present in spatial_raw.
pair_markers = {m for col in adata.obsm['spatial_raw'].columns for m in col.split('/')}
panel = json.load(open(NEW_DATA / 'marker_panels.json'))['cd8_t_cell_markers']
panel = [m for grp in panel.values() for m in grp] if isinstance(panel, dict) else panel
markers_tri = list(dict.fromkeys(m for m in panel if m in pair_markers))
n_valid = len(get_marker_triplets(markers_tri, adata.obsm['spatial_raw'].columns))
print(f'{len(markers_tri)} panel markers present in spatial_raw -> {n_valid} valid triplets')

# determinant: 1 - det(R) of the 3x3 pseudo-correlation matrix (Gaussian total correlation)
TRIPLET_KEY = build_triplet_obsm(
    adata, source_key='spatial_raw', markers=markers_tri,
    metric='determinant', value_transform='asinh', top_k=300,
)
print(adata.obsm[TRIPLET_KEY].shape)

In [ ]:
setup_kwargs_tri = dict(
    layer=ABUNDANCE_LAYER,
    extra_modality_keys=[SPATIAL_KEY, TRIPLET_KEY],  # abundance + pairwise + 3-way
    n_modalities=3,
    batch_key=BATCH_KEY,
    spatial_mask_key=None,
)
model_kwargs_tri = dict(
    n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal, D.Normal, D.Normal],
    experts_method='POE',
    loss_weights='auto',
    joint_kl=False, unimodal_kl=True,
    batch_mask=[False, True, True],  # batch-corr pairwise + triplet
)
model_tri = train_model(adata, MultiModalSCVI, setup_kwargs_tri, model_kwargs_tri, base_train_kwargs)
model_tri.save(str(MODELS / 'poe_joint_triplet'), overwrite=True)
print('triplet weights:', model_tri.get_weights())  # 3 modality weights: abundance / pairwise / triplet